# 03 转化漏斗 + 品类拆解

回答：**用户怎么买东西？哪里卡住了？不同品类差异多大？**

In [ ]:
import sys; sys.path.append('..')
from utils.db import query
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
df = query("SELECT * FROM user_behavior_clean WHERE date < '2017-12-08'")

## 1. 行为渗透率

用户行为不是严格顺序的（用户可以跳过收藏直接加购），所以这里不是"漏斗"，而是**各行为的覆盖度**——在所有活跃用户中，有百分之多少做过每种行为。

In [ ]:
# 行为渗透率：每种行为的用户覆盖度
total_uv = df['user_id'].nunique()
behaviors = ['pv', 'fav', 'cart', 'buy']
labels = ['浏览', '收藏', '加购', '购买']
penetration = pd.DataFrame({
    'stage': labels,
    'users': [df[df['behavior_type']==b]['user_id'].nunique() for b in behaviors]
})
penetration['rate'] = (penetration['users'] / total_uv * 100).round(1)
print(f'总UV: {total_uv:,}')
print(penetration)

In [ ]:
# 关键：在 PV 用户中，有多少也做了其他行为？（这才是真正的"转化路径"）
pv_set = set(df[df['behavior_type']=='pv']['user_id'].unique())
fav_set = set(df[df['behavior_type']=='fav']['user_id'].unique())
cart_set = set(df[df['behavior_type']=='cart']['user_id'].unique())
buy_set = set(df[df['behavior_type']=='buy']['user_id'].unique())

overlap = pd.DataFrame({
    '行为': ['浏览(pv)', '其中也收藏', '其中也加购', '其中也购买'],
    '用户数': [
        len(pv_set),
        len(pv_set & fav_set),
        len(pv_set & cart_set),
        len(pv_set & buy_set),
    ]
})
overlap['占PV用户比例'] = (overlap['用户数'] / len(pv_set) * 100).round(1)
print('行为渗透率（以浏览用户为基准）:')
print(overlap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：行为渗透条形图
colors = ['#4C72B0', '#55A868', '#DD8452', '#C44E52']
ax = axes[0]
bars = ax.bar(penetration['stage'], penetration['users'], color=colors)
ax.set_title('各行为的用户覆盖度')
ax.set_ylabel('用户数')
for bar, rate in zip(bars, penetration['rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{rate}%', ha='center', fontweight='bold')

# 右图：PV 用户中各行为的渗透
ax = axes[1]
ov = overlap.iloc[1:]  # skip the pv row itself
bars = ax.bar(ov['行为'], ov['占PV用户比例'], color=['#55A868', '#DD8452', '#C44E52'])
ax.set_ylabel('占PV用户比例 (%)')
ax.set_title('浏览用户中，也做了其他行为的比例')
for bar, v in zip(bars, ov['占PV用户比例']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/fig_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. 分品类漏斗对比

不同品类的转化行为可能天差地别——高频低客单价 vs 低频高客单价，漏斗形状完全不同。

In [ ]:
# 用 SQL 做分品类漏斗（窗口函数 + CASE WHEN）
category_funnel_sql = """
WITH cat_behavior AS (
    SELECT category_id, behavior_type, COUNT(DISTINCT user_id) AS uv
    FROM user_behavior_clean
    WHERE date < '2017-12-08'
    GROUP BY category_id, behavior_type
)
SELECT 
    category_id,
    MAX(CASE WHEN behavior_type='pv' THEN uv END) AS pv_users,
    MAX(CASE WHEN behavior_type='fav' THEN uv END) AS fav_users,
    MAX(CASE WHEN behavior_type='cart' THEN uv END) AS cart_users,
    MAX(CASE WHEN behavior_type='buy' THEN uv END) AS buy_users
FROM cat_behavior
GROUP BY category_id
"""
cat_funnel = query(category_funnel_sql)
cat_funnel['purchase_rate'] = (cat_funnel['buy_users'] / cat_funnel['pv_users'] * 100).round(2)
cat_funnel = cat_funnel.sort_values('purchase_rate', ascending=False)
cat_funnel = cat_funnel[cat_funnel['pv_users'] > 50]  # 过滤样本太少的品类
print(f'有效品类数: {len(cat_funnel)}')
print(f'\n转化率最高 Top 5 品类:')
print(cat_funnel.head(5)[['category_id', 'pv_users', 'buy_users', 'purchase_rate']])
print(f'\n转化率最低 Bottom 5 品类:')
print(cat_funnel.tail(5)[['category_id', 'pv_users', 'buy_users', 'purchase_rate']])
print(f'\n品类转化率分布: 均值={cat_funnel["purchase_rate"].mean():.1f}%, 中位数={cat_funnel["purchase_rate"].median():.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(cat_funnel['purchase_rate'], bins=30, color='#4C72B0', alpha=0.7, edgecolor='white')
ax.axvline(cat_funnel['purchase_rate'].mean(), color='red', linestyle='--',
           label=f'均值={cat_funnel["purchase_rate"].mean():.1f}%')
ax.axvline(cat_funnel['purchase_rate'].median(), color='green', linestyle='--',
           label=f'中位数={cat_funnel["purchase_rate"].median():.1f}%')
ax.set_xlabel('品类购买转化率 (%)')
ax.set_ylabel('品类数')
ax.set_title('各品类 PV→购买 转化率分布')
ax.legend()
plt.tight_layout()
plt.savefig('../data/fig_category_funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 用户路径分析

用户实际走了哪些路径？不是所有用户都走 pv→fav→cart→buy 标准流程。

In [ ]:
# 每个用户在每个商品上的行为聚合为路径
path_sql = """
SELECT 
    user_id, item_id,
    GROUP_CONCAT(DISTINCT behavior_type ORDER BY time) AS path
FROM user_behavior_clean
WHERE date < '2017-12-08'
GROUP BY user_id, item_id
"""
paths = query(path_sql)
print(f'总共 {len(paths):,} 个用户-商品交互对')

# 统计 Top 路径
path_counts = paths['path'].value_counts().head(10)
path_pct = (path_counts / len(paths) * 100).round(2)
print(f'\nTop 10 用户路径:')
print(pd.DataFrame({'次数': path_counts, '占比(%)': path_pct}))

In [ ]:
# 定义关键路径类型
def classify_path(p):
    if p == 'pv': return '仅浏览'
    if 'buy' in p and 'cart' in p: return '加购后购买'
    if 'buy' in p and 'fav' in p and 'cart' not in p: return '收藏后购买'
    if 'buy' in p and 'fav' not in p and 'cart' not in p: return '直接购买(浏览→买)'
    if 'cart' in p and 'buy' not in p: return '加购未买'
    if 'fav' in p and 'cart' not in p and 'buy' not in p: return '收藏未买'
    return '其他'

paths['path_type'] = paths['path'].apply(classify_path)
path_type_counts = paths['path_type'].value_counts()
print('用户路径类型分布:')
print(path_type_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_path = ['#4C72B0', '#55A868', '#DD8452', '#C44E52', '#8E44AD', '#E67E22', '#1ABC9C']
axes[0].pie(path_type_counts.values, labels=path_type_counts.index,
            autopct='%1.1f%%', colors=colors_path, startangle=90)
axes[0].set_title('用户路径类型')

# 有购买的路径中，加购 vs 直接买 vs 收藏
buy_paths = paths[paths['path'].str.contains('buy')]
buy_types = buy_paths['path_type'].value_counts()
axes[1].bar(range(len(buy_types)), buy_types.values, color=['#C44E52', '#DD8452', '#8E44AD'])
axes[1].set_xticks(range(len(buy_types)))
axes[1].set_xticklabels(buy_types.index, rotation=15)
axes[1].set_title('购买用户的路径拆解')
for i, v in enumerate(buy_types.values):
    axes[1].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/fig_paths.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 关键发现

In [ ]:
pv_only = paths[paths['path'] == 'pv'].shape[0]
cart_no_buy = paths[paths['path_type'] == '加购未买'].shape[0]
buy_penetration = len(pv_set & buy_set) / len(pv_set) * 100
print('=== 行为渗透与路径分析关键发现 ===')
print(f'1. 浏览用户中，最终购买的比例: {buy_penetration:.1f}%')
print(f'2. 约 {pv_only/len(paths)*100:.1f}% 的用户-商品交互对止步于浏览')
print(f'3. 品类间购买转化率差异大: 最高 {cat_funnel["purchase_rate"].max():.1f}% vs 最低 {cat_funnel["purchase_rate"].min():.1f}%')
print(f'4. 加购但未买的用户-商品对有 {cart_no_buy:,} 对——可挽回空间')
print(f'5. 加购→购买是最高效的转化路径，收藏→购买的转化率较低')